In [ ]:
import pandas as pd
import numpy as np

# ---- Input ----
FLIGHTS_CSV = "opensky_flights_yesterday.csv"   # produced by get_data_from_open_sky.ipynb
AIRPORT_FILTER = None  # e.g., "LHBP" (Budapest). Set None to keep all airports.

# ---- Turnaround pairing constraints ----
MIN_TA_MINUTES = 15      # ignore impossible short turnarounds
MAX_TA_HOURS   = 8       # ignore very long gaps (overnight / maintenance) unless you want them

# ---- Proxy offsets for on/off-block (in minutes) ----
EPS_ONBLOCK_MIN = 2
EPS_OFFBLOCK_MIN = 2

df = pd.read_csv(FLIGHTS_CSV)
print("Flights loaded:", len(df))
print("Columns:", list(df.columns))


In [ ]:

# Basic cleaning + timestamps
needed = ["icao24","firstSeen","lastSeen","estDepartureAirport","estArrivalAirport","callsign"]
for c in needed:
    if c not in df.columns:
        raise ValueError(f"Missing required column: {c}")

df["firstSeen"] = pd.to_numeric(df["firstSeen"], errors="coerce")
df["lastSeen"]  = pd.to_numeric(df["lastSeen"], errors="coerce")

df = df.dropna(subset=["icao24","firstSeen","lastSeen","estDepartureAirport","estArrivalAirport"])
df["firstSeen_dt"] = pd.to_datetime(df["firstSeen"].astype(int), unit="s", utc=True)
df["lastSeen_dt"]  = pd.to_datetime(df["lastSeen"].astype(int), unit="s", utc=True)

# Optional: focus on one airport
if AIRPORT_FILTER:
    df = df[(df["estDepartureAirport"] == AIRPORT_FILTER) | (df["estArrivalAirport"] == AIRPORT_FILTER)].copy()
    print("After airport filter:", len(df))

# Keep only sensible flights (arrival after departure)
df = df[df["lastSeen_dt"] >= df["firstSeen_dt"]].copy()

# Normalize callsign
df["callsign"] = df["callsign"].astype(str).str.strip().replace({"nan": np.nan})

df = df.sort_values(["icao24","firstSeen_dt"]).reset_index(drop=True)
df.head()


In [ ]:

# Pair consecutive flights per aircraft to create turnarounds
pairs = []
for icao, g in df.groupby("icao24", sort=False):
    g = g.sort_values("firstSeen_dt")
    g_next = g.shift(-1)

    # A valid turnaround happens at the airport where inbound arrives and outbound departs
    same_airport = (g["estArrivalAirport"] == g_next["estDepartureAirport"])

    # turnaround duration between inbound arrival and outbound departure
    ta_minutes = (g_next["firstSeen_dt"] - g["lastSeen_dt"]).dt.total_seconds() / 60.0

    ok_window = (ta_minutes >= MIN_TA_MINUTES) & (ta_minutes <= MAX_TA_HOURS * 60)

    valid = same_airport & ok_window

    gg = pd.DataFrame({
        "icao24": g["icao24"],
        "in_callsign": g["callsign"],
        "out_callsign": g_next["callsign"],
        "in_origin": g["estDepartureAirport"],
        "turnaround_airport": g["estArrivalAirport"],
        "out_destination": g_next["estArrivalAirport"],
        "in_firstSeen_dt": g["firstSeen_dt"],
        "in_lastSeen_dt": g["lastSeen_dt"],
        "out_firstSeen_dt": g_next["firstSeen_dt"],
        "out_lastSeen_dt": g_next["lastSeen_dt"],
        "turnaround_minutes": ta_minutes
    })
    gg = gg[valid].dropna(subset=["turnaround_airport","out_destination"]).copy()
    pairs.append(gg)

turnarounds = pd.concat(pairs, ignore_index=True) if pairs else pd.DataFrame()
print("Turnarounds built:", len(turnarounds))
turnarounds.head()


In [ ]:

# Create turnaround event log (CSV) with proxy milestones
if turnarounds.empty:
    raise RuntimeError("No turnarounds were formed. Try widening the time window or removing airport filter.")

eps_on  = pd.to_timedelta(EPS_ONBLOCK_MIN, unit="m")
eps_off = pd.to_timedelta(EPS_OFFBLOCK_MIN, unit="m")

# Case id: aircraft + airport + inbound arrival time
turnarounds["case:concept:name"] = (
    turnarounds["icao24"].astype(str) + "_" +
    turnarounds["turnaround_airport"].astype(str) + "_" +
    turnarounds["in_lastSeen_dt"].dt.strftime("%Y%m%dT%H%M%SZ")
)

events = []

def add_events(activity, ts_col):
    e = pd.DataFrame({
        "case:concept:name": turnarounds["case:concept:name"],
        "concept:name": activity,
        "time:timestamp": turnarounds[ts_col],
        "icao24": turnarounds["icao24"],
        "turnaround_airport": turnarounds["turnaround_airport"],
        "in_callsign": turnarounds["in_callsign"],
        "out_callsign": turnarounds["out_callsign"],
        "in_origin": turnarounds["in_origin"],
        "out_destination": turnarounds["out_destination"],
        "turnaround_minutes": turnarounds["turnaround_minutes"],
    })
    return e

events.append(add_events("ARRIVAL", "in_lastSeen_dt"))
events.append(add_events("ON_BLOCK", "in_lastSeen_dt") .assign(**{"time:timestamp": turnarounds["in_lastSeen_dt"] + eps_on}))
events.append(add_events("OFF_BLOCK", "out_firstSeen_dt").assign(**{"time:timestamp": turnarounds["out_firstSeen_dt"] - eps_off}))
events.append(add_events("DEPARTURE", "out_firstSeen_dt"))

event_log = pd.concat(events, ignore_index=True)
event_log = event_log.dropna(subset=["time:timestamp"]).sort_values(["case:concept:name","time:timestamp"]).reset_index(drop=True)

# Save
OUT_CSV = "turnaround_eventlog_opensky.csv"
event_log.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV)
event_log.head(12)


In [ ]:

# Quick sanity checks (useful for Milestone 3 validation writeup)
case_counts = event_log.groupby("case:concept:name")["concept:name"].nunique()
print("Cases:", case_counts.size)
print("Activities per case (unique):")
print(case_counts.value_counts().sort_index())

# Turnaround duration distribution
ta = turnarounds["turnaround_minutes"].dropna()
print("\nTurnaround minutes summary:")
print(ta.describe(percentiles=[0.05,0.25,0.5,0.75,0.95]))

# Optional: basic plot
import matplotlib.pyplot as plt
plt.figure()
plt.hist(ta, bins=40)
plt.xlabel("Turnaround duration (minutes)")
plt.ylabel("Count")
plt.title("OpenSky-derived turnaround durations")
plt.show()


In [ ]:

# Optional: export to XES for pm4py (if installed)
try:
    import pm4py
    from pm4py.objects.conversion.log import converter as log_converter
    from pm4py.objects.log.util import dataframe_utils
    from pm4py.objects.log.exporter.xes import exporter as xes_exporter

    df_ev = event_log.copy()
    df_ev["time:timestamp"] = pd.to_datetime(df_ev["time:timestamp"], utc=True)
    df_ev = dataframe_utils.convert_timestamp_columns_in_df(df_ev)

    log = log_converter.apply(df_ev, variant=log_converter.Variants.TO_EVENT_LOG)
    OUT_XES = "turnaround_eventlog_opensky.xes"
    xes_exporter.apply(log, OUT_XES)
    print("Saved:", OUT_XES)
except Exception as e:
    print("pm4py export skipped:", e)
